# 01c · Data figures (Fig. 3, Fig. 4, Fig. 5)
Drawn from the prepared data. Outputs go to `results/figures/`.

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        break
print("project root:", ROOT)

In [ ]:
import numpy as np
from IPython.display import Image, display
from fuguard_repro.config import image_config, awe_config
from fuguard_repro.data_image import client_class_counts, test_class_counts
from fuguard_repro import plotting

cfg = image_config("SVHN")
before = client_class_counts(cfg, 0, before_injection=True)
after = client_class_counts(cfg, 0, before_injection=False)
p3 = plotting.plot_label_distribution(before, after, "SVHN, 10 clients")
clean, poison = test_class_counts(cfg, 0)
p4 = plotting.plot_test_distribution(clean, poison, "SVHN test sets")
display(Image(str(p3))); display(Image(str(p4)))

### Fig. 5 · AWE sample before / after trigger (needs 01b)

In [ ]:
from fuguard_repro.data_awe import awe_dir
d = np.load(awe_dir(awe_config()) / "global_test.npz")
i = 0   # first non-target test window; X_backdoor[i] is its triggered version
mask = d["y_clean"] != awe_config()["target_label"]
clean_w, trig_w = d["X_clean"][mask][i][0], d["X_backdoor"][i][0]     # first row of the 30x1000 window
display(Image(str(plotting.plot_awe_trigger(clean_w, trig_w))))